In [ ]:
import numpy as np
import diplib as dip
from tifffile import tifffile as tiff

IN = "10-head10x_img0.tiff"          
OUT = "10-head10x_img0_processed.tiff"

img = dip.ImageReadTIFF(IN, slice(0, -1))
print("sizes:", img.Sizes(), "| channels:", img.TensorElements())

def process(ch):
    ch = dip.Convert(ch, "SFLOAT")
    ch = dip.MedianFilter(ch, dip.Kernel([3, 3, 3]))   # remove spikes
    ch = dip.Tophat(ch, dip.SE(5, "elliptic"))         # remove bg
    ch = dip.BilateralFilter(ch, spatialSigmas=1.0, tonalSigma=30.0)  # denoise, keep edges
    return ch

channels = [process(img(c)) for c in range(img.TensorElements())]
stack = np.stack([np.asarray(dip.Convert(c, "UINT16")) for c in channels], axis=1)  # (Z,C,Y,X)

tiff.imwrite(OUT, stack, imagej=True, metadata={"axes": "ZCYX"})
print(f"saved {OUT}  shape={stack.shape} (Z,C,Y,X)")


sizes: [512, 512, 43] | channels: 3
saved Series016_decon_denoised.tif  shape=(43, 3, 512, 512) (Z,C,Y,X)
